# US Visa (OFLC Labor Certification) — Revised EDA
**What changed vs. the original notebook**
1. `prevailing_wage` is converted to a **common yearly scale** (the original mixed hourly / weekly / monthly / yearly values).
2. Data-quality checks added: negative `no_of_employees`, implausible `yr_of_estab`, duplicates.
3. Target imbalance stated correctly (Certified ≈ 67 %, Denied ≈ 33 %).
4. Chi-square no longer tests the target against itself; **Cramér's V** (effect size) added.
5. Real multicollinearity checks (numeric: correlation + VIF, categorical: Cramér's V matrix).
6. Continents are **not** blindly merged; category sizes *and* approval rates are shown with confidence intervals.
7. Extra EDA: interactions, mutual information, baseline model + permutation importance, fairness audit.

Dataset: https://www.kaggle.com/datasets/moro23/easyvisa-dataset (`Visadataset.csv`)

## 1. Setup & load

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_PATH = 'Visadataset.csv'   # <- change if needed
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Data-quality audit (missing in the original)

In [ ]:
print('Duplicate rows (ignoring case_id):', df.drop(columns='case_id').duplicated().sum())
print('case_id unique:', df.case_id.is_unique)
print('Missing values:', df.isna().sum().sum())
print()
print('Negative no_of_employees:', (df.no_of_employees < 0).sum())
print(df.loc[df.no_of_employees < 0, 'no_of_employees'].describe())
print()
print('yr_of_estab < 1900:', (df.yr_of_estab < 1900).sum(), '| range:', df.yr_of_estab.min(), '-', df.yr_of_estab.max())
print()
df.describe()

**Findings to confirm from the output above:** `no_of_employees` has impossible negative values (min was −26 in the original). They are almost certainly sign errors, so we take the absolute value and keep a flag. `yr_of_estab` runs back to 1800, so we use *company age* (relative to the latest year in the data) instead of the raw year.

In [ ]:
df['emp_was_negative'] = (df.no_of_employees < 0).astype(int)
df['no_of_employees'] = df.no_of_employees.abs()

REF_YEAR = df.yr_of_estab.max()            # dataset appears to be from ~2016
df['company_age'] = REF_YEAR - df.yr_of_estab
df['log_employees'] = np.log1p(df.no_of_employees)

df['target'] = (df.case_status == 'Denied').astype(int)   # 1 = Denied (the minority / 'positive' class)

### 2.1 Fix `prevailing_wage`: convert everything to a yearly figure

In [ ]:
print(df.groupby('unit_of_wage').prevailing_wage.describe())

The raw column clearly mixes scales (hourly values are a few to a few hundred, yearly values are tens of thousands). Comparing them directly is meaningless.

In [ ]:
ANNUAL_FACTOR = {'Hour': 2080, 'Week': 52, 'Month': 12, 'Year': 1}   # 40 h x 52 wk
df['yearly_wage'] = df.prevailing_wage * df.unit_of_wage.map(ANNUAL_FACTOR)
df['log_yearly_wage'] = np.log1p(df.yearly_wage)

print(df.groupby('unit_of_wage').yearly_wage.describe())
print('\nRows with yearly_wage > $500k after conversion:', (df.yearly_wage > 500_000).sum())
print('Rows with yearly_wage < $10k after conversion :', (df.yearly_wage < 10_000).sum())

**Sanity check:** after conversion, inspect the two counts above. A few hourly rows may convert to implausibly large yearly wages (probable entry errors). Decide explicitly whether to cap them (winsorize) or leave them. Tree models tolerate this; linear models will not. The `log_yearly_wage` column dampens the effect either way.

## 3. Target variable

In [ ]:
vc = df.case_status.value_counts()
print(vc, '\n', (vc/len(df)*100).round(2))
fig, ax = plt.subplots(figsize=(5,5))
ax.pie(vc, labels=vc.index, autopct='%1.1f%%', colors=['#1188ff','#e63a2a'], startangle=90, explode=(0,.06))
ax.set_title('case_status'); plt.show()

**Certified ≈ 66.8 %, Denied ≈ 33.2 %** — a *mild* imbalance (roughly 2:1), and the **majority class is Certified** (the original notebook said the opposite). Consequences: use a **stratified** split, report F1/recall/PR-AUC for *Denied*, and use class weights before reaching for resampling. Any resampling must happen only on the training split.

## 4. Univariate analysis

In [ ]:
num_cols = ['no_of_employees', 'company_age', 'yearly_wage']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, c in enumerate(num_cols):
    sns.histplot(df[c], bins=50, ax=axes[0, i]); axes[0, i].set_title(f'{c} (raw)')
log_versions = {'no_of_employees': np.log1p(df.no_of_employees), 'company_age': df.company_age, 'yearly_wage': np.log1p(df.yearly_wage)}
for i, (c, s) in enumerate(log_versions.items()):
    sns.histplot(s, bins=50, ax=axes[1, i], color='seagreen'); axes[1, i].set_title(f'{c} ({"log1p" if c!="company_age" else "as is"})')
plt.tight_layout(); plt.show()
print(df[num_cols].skew().round(2))

In [ ]:
cat_cols = ['continent','education_of_employee','has_job_experience','requires_job_training',
            'region_of_employment','unit_of_wage','full_time_position']
fig, axes = plt.subplots(2, 4, figsize=(20, 9)); axes = axes.ravel()
for ax, c in zip(axes, cat_cols):
    order = df[c].value_counts().index
    sns.countplot(x=c, data=df, order=order, ax=ax); ax.tick_params(axis='x', rotation=30)
    ax.set_title(f'{c}')
axes[-1].axis('off'); plt.tight_layout(); plt.show()
for c in cat_cols:
    print((df[c].value_counts(normalize=True)*100).round(2).to_string(), '\n' + '-'*30)

**Reading this correctly:** a dominant category (e.g. `Year` = 90 % of `unit_of_wage`) only means low variance in that feature. It says **nothing** about importance. Importance is judged from the relationship with the target (next section).

## 5. Categorical features vs. target

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / d
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return centre - half, centre + half

def rate_table(col):
    g = df.groupby(col).agg(n=('target','size'), denied=('target','sum'))
    g['pct_of_data'] = g.n / len(df) * 100
    g['certified_rate'] = (1 - g.denied / g.n) * 100
    ci = [wilson(n - d, n) for n, d in zip(g.n, g.denied)]
    g['ci_low'] = [c[0]*100 for c in ci]; g['ci_high'] = [c[1]*100 for c in ci]
    return g.drop(columns='denied').sort_values('certified_rate', ascending=False)

def plot_rate(col, ax):
    t = rate_table(col)
    ax.bar(t.index.astype(str), t.certified_rate,
           yerr=[t.certified_rate - t.ci_low, t.ci_high - t.certified_rate], capsize=4, color='#1188ff', alpha=.8)
    ax.axhline((1-df.target.mean())*100, color='red', ls='--', label='overall')
    for i, (r, n) in enumerate(zip(t.certified_rate, t.n)):
        ax.text(i, r + 1.5, f'{r:.1f}%\nn={n:,}', ha='center', fontsize=8)
    ax.set_ylim(0, 105); ax.set_title(f'Certified rate by {col}'); ax.tick_params(axis='x', rotation=20)

fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.ravel()
for ax, c in zip(axes, cat_cols): plot_rate(c, ax)
axes[0].legend(); axes[-1].axis('off'); plt.tight_layout(); plt.show()

In [ ]:
for c in cat_cols:
    print(f'=== {c} ==='); print(rate_table(c).round(1).to_string(), '\n')

**Notes on how to read the bars:** error bars are 95 % Wilson confidence intervals. Wide bars (Oceania, Month, Week, Island) mean small samples, so treat those differences cautiously.

From the original outputs, still valid: Education (Doctorate 87 %, Master's 79 %, Bachelor's 62 %, High School 34 %), Experience (Y 74.5 % vs N 56.1 %, an 18-point gap: strong, not "slight"), Continent (Europe 79 % … South America 58 %), Region (Midwest 75.5 %, South 70 % vs West/Northeast/Island ≈ 60–63 %: **region does matter**), and `requires_job_training` (66.6 % vs 67.9 %: negligible).

### 5.1 Association strength: chi-square **and** Cramér's V (target excluded from the loop)

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2/n - (k-1)*(r-1)/(n-1))            # bias-corrected
    rc, kc = r - (r-1)**2/(n-1), k - (k-1)**2/(n-1)
    return np.sqrt(phi2 / max(1e-12, min(kc-1, rc-1)))

rows = []
for c in cat_cols:                                       # note: case_status NOT in this list
    ct = pd.crosstab(df[c], df.case_status)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    rows.append([c, chi2, dof, p, cramers_v(df[c], df.case_status)])
assoc = pd.DataFrame(rows, columns=['feature','chi2','dof','p_value','cramers_v']).sort_values('cramers_v', ascending=False)
assoc['verdict'] = np.where(assoc.p_value < 0.05, 'associated', 'no evidence of association')
assoc['effect'] = pd.cut(assoc.cramers_v, [-1, .1, .2, .3, 1], labels=['negligible','small','moderate','large'])
assoc

With ~25 k rows nearly everything reaches p < 0.05, so **Cramér's V (effect size)** is the number to rank on. "Fail to reject" for `requires_job_training` means *no evidence of association*, not proof of independence. Keep it, or drop it only after checking it with a model.

## 6. Numeric features vs. target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_vars = [('log_employees','log(1 + employees)'), ('company_age','company age (years)'), ('log_yearly_wage','log(1 + yearly wage)')]
for i, (c, lab) in enumerate(plot_vars):
    sns.boxplot(data=df, x='case_status', y=c, palette=['#1188ff','#e63a2a'], ax=axes[0,i]); axes[0,i].set_title(lab)
    sns.kdeplot(data=df, x=c, hue='case_status', common_norm=False, palette=['#1188ff','#e63a2a'], ax=axes[1,i])
plt.tight_layout(); plt.show()

rows = []
for c in ['no_of_employees','company_age','yearly_wage']:
    a = df.loc[df.target==0, c]; b = df.loc[df.target==1, c]
    u, p = stats.mannwhitneyu(a, b)
    auc = u / (len(a)*len(b))                              # P(Certified value > Denied value)
    rows.append([c, a.median(), b.median(), p, auc])
pd.DataFrame(rows, columns=['feature','median_certified','median_denied','mannwhitney_p','prob_cert_gt_denied'])

`prob_cert_gt_denied` ≈ 0.5 means the feature does not separate the classes; the further from 0.5, the stronger the univariate signal. Non-parametric tests are used because these variables are heavily skewed.

## 7. Wage analysis — redone on the yearly scale

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.boxplot(data=df, y='education_of_employee', x='yearly_wage', ax=axes[0], showfliers=False,
            order=['High School',"Bachelor's","Master's",'Doctorate']); axes[0].set_title('Yearly prevailing wage by education')
sns.boxplot(data=df, y='continent', x='yearly_wage', ax=axes[1], showfliers=False); axes[1].set_title('by continent')
sns.boxplot(data=df, y='has_job_experience', x='yearly_wage', ax=axes[2], showfliers=False); axes[2].set_title('by job experience')
plt.tight_layout(); plt.show()
print(df.groupby('education_of_employee').yearly_wage.agg(['median','mean','count']).round(0))
print(df.groupby('has_job_experience').yearly_wage.agg(['median','mean','count']).round(0))
print(df.groupby('continent').yearly_wage.agg(['median','mean','count']).round(0))

Re-read every wage claim from the original against these tables (education ordering, "experience earns less", "Asia earns more"). Those statements came from mixed-unit data. Remember also that `prevailing_wage` is the **Department of Labor benchmark wage for the job/location**, not the applicant's own salary, so phrase conclusions as "jobs with a higher prevailing wage…".

In [ ]:
# Does wage level relate to approval? Use quantile bins of YEARLY wage, and show the unit_of_wage effect separately
df['wage_bin'] = pd.qcut(df.yearly_wage, 10, duplicates='drop')
t = df.groupby('wage_bin').target.agg(['size','mean']); t['certified_rate'] = (1-t['mean'])*100
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
t.certified_rate.plot(kind='bar', ax=ax[0], color='#1188ff'); ax[0].set_title('Certified rate by yearly-wage decile'); ax[0].tick_params(axis='x', rotation=45)
ax[0].axhline((1-df.target.mean())*100, color='red', ls='--')
u = df.groupby('unit_of_wage').agg(n=('target','size'), cert=('target', lambda s: (1-s.mean())*100))
u.cert.plot(kind='bar', ax=ax[1], color='#1188ff'); ax[1].set_title('Certified rate by wage unit'); ax[1].axhline((1-df.target.mean())*100, color='red', ls='--')
plt.tight_layout(); plt.show(); print(u.round(1))

## 8. Multicollinearity — proper checks

In [ ]:
# 8.1 numeric: Spearman correlation + VIF (computed manually, no statsmodels needed)
from sklearn.linear_model import LinearRegression
X = df[['log_employees','company_age','log_yearly_wage']]
corr = X.corr(method='spearman')
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1); plt.title('Spearman correlation'); plt.show()

def vif(X):
    out = {}
    for c in X.columns:
        r2 = LinearRegression().fit(X.drop(columns=c), X[c]).score(X.drop(columns=c), X[c])
        out[c] = 1 / (1 - r2)
    return pd.Series(out, name='VIF')
vif(X)

In [ ]:
# 8.2 categorical vs categorical: Cramér's V matrix (feature-vs-feature, this is the actual multicollinearity check)
M = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        M.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])
plt.figure(figsize=(9,7)); sns.heatmap(M, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1)
plt.title("Cramér's V between categorical features"); plt.show()

Rules of thumb: |ρ| > 0.8 or VIF > 5–10 signals a problem. For Cramér's V, > 0.5 between two features is worth a look. Interpret the three printed numeric VIFs directly instead of assuming none exist.

## 9. Additional EDA that adds real value

### 9.1 Interactions: education × experience

In [ ]:
pt = df.pivot_table(index='education_of_employee', columns='has_job_experience', values='target', aggfunc=lambda s: (1-s.mean())*100)
cnt = df.pivot_table(index='education_of_employee', columns='has_job_experience', values='target', aggfunc='size')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(pt, annot=True, fmt='.1f', cmap='Blues', ax=ax[0]); ax[0].set_title('Certified rate (%)')
sns.heatmap(cnt, annot=True, fmt='d', cmap='Greys', ax=ax[1]); ax[1].set_title('Row counts')
plt.tight_layout(); plt.show()

### 9.2 Interactions: wage unit × education, and region × education

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for a, (r, cname) in zip(ax, [('unit_of_wage','education_of_employee'), ('region_of_employment','education_of_employee')]):
    p = df.pivot_table(index=r, columns=cname, values='target', aggfunc=lambda s: (1-s.mean())*100)
    sns.heatmap(p, annot=True, fmt='.0f', cmap='Blues', ax=a); a.set_title(f'Certified rate: {r} × {cname}')
plt.tight_layout(); plt.show()

### 9.3 Company age & size bins

In [ ]:
df['age_bin'] = pd.cut(df.company_age, [-1, 5, 10, 20, 50, 100, 300])
df['size_bin'] = pd.qcut(df.no_of_employees, 6, duplicates='drop')
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for a, b in zip(ax, ['age_bin','size_bin']):
    t = df.groupby(b).target.agg(['size','mean']); (1-t['mean']).mul(100).plot(kind='bar', ax=a, color='#1188ff')
    a.axhline((1-df.target.mean())*100, color='red', ls='--'); a.set_title(f'Certified rate by {b}'); a.set_ylabel('%')
plt.tight_layout(); plt.show()

### 9.4 Mutual information (captures non-linear / categorical signal in one ranking)

In [ ]:
from sklearn.feature_selection import mutual_info_classif
feat = cat_cols + ['log_employees','company_age','log_yearly_wage']
Xm = df[feat].copy()
for c in cat_cols: Xm[c] = Xm[c].astype('category').cat.codes
mi = mutual_info_classif(Xm, df.target, discrete_features=[c in cat_cols for c in feat], random_state=0)
pd.Series(mi, index=feat).sort_values().plot(kind='barh', figsize=(7,5), title='Mutual information with Denied'); plt.show()

### 9.5 Baseline model: stratified split, permutation importance (a reality check on the EDA)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

num_feat = ['log_employees','company_age','log_yearly_wage']
X_tr, X_te, y_tr, y_te = train_test_split(df[cat_cols+num_feat], df.target, test_size=.2, stratify=df.target, random_state=42)
pipe = Pipeline([('prep', ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')),
                 ('rf', RandomForestClassifier(300, min_samples_leaf=10, class_weight='balanced', n_jobs=-1, random_state=42))])
pipe.fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:,1]
print('ROC-AUC (Denied):', round(roc_auc_score(y_te, proba), 3))
print(classification_report(y_te, pipe.predict(X_te), target_names=['Certified','Denied']))

pi = permutation_importance(pipe, X_te, y_te, scoring='roc_auc', n_repeats=8, random_state=0, n_jobs=-1)
pd.Series(pi.importances_mean, index=X_te.columns).sort_values().plot(kind='barh', figsize=(7,5), title='Permutation importance (ΔAUC)'); plt.show()

Compare this ranking against Cramér's V and mutual information. Features that look weak in EDA but rank high here usually act through **interactions**; features that look strong in EDA but are unimportant here are usually redundant with another feature.

### 9.6 Fairness audit (continent is a proxy for nationality)

In [ ]:
te = X_te.copy(); te['y'] = y_te.values; te['p'] = proba; te['pred'] = (proba > .5).astype(int)
audit = te.groupby('continent').agg(n=('y','size'), actual_denied=('y','mean'), predicted_denied=('pred','mean'))
audit['gap'] = audit.predicted_denied - audit.actual_denied
audit.round(3)

If the model flags one continent as Denied far more often than the base rate justifies, or if the feature is legally or ethically off-limits for this use, drop `continent` and check how much performance you lose. The INA prohibits discrimination based on national origin, so this decision should be explicit, not incidental.

### 9.7 Other checks worth doing
- **Missing time dimension:** the data has no application date, so you can't check for drift. Note it as a limitation. Random splits may be optimistic.
- **Outlier treatment by model type:** tree models need none; for linear/NN models winsorize `yearly_wage` at, say, the 1st/99th percentile.
- **Rare categories:** `Island` region, `Month`/`Week` wage units and `Oceania` have very few rows; use regularised or target encoding and check stability with cross-validation.
- **Leakage review:** confirm every feature exists *before* the decision is made (e.g. `prevailing_wage` is set by DOL before adjudication, which is fine).
- **Calibration:** if probabilities will drive shortlisting, check a calibration curve, not just AUC.

## 10. Corrected summary
| # | Finding |
|---|---|
| 1 | Target: **Certified 66.8 % / Denied 33.2 %**; mild imbalance, majority = Certified. |
| 2 | `no_of_employees` had negative values (fixed via `abs`); `yr_of_estab` is better used as company age. |
| 3 | `prevailing_wage` must be annualised; all wage-based conclusions from the original notebook are void. |
| 4 | Strong predictors (from category rates): **education, job experience**. Moderate: **region, continent, wage unit**. Weak/none: `requires_job_training`. Rank the rest by Cramér's V from section 5.1. |
| 5 | Do **not** merge Europe / N. America / S. America into "Others"; they differ by up to 21 points. Merge only truly rare categories, if at all. |
| 6 | Handle imbalance through stratified split, class weights and appropriate metrics, not accuracy. |
| 7 | Confirm multicollinearity with sections 8.1–8.2, and compare EDA rankings with the baseline model in 9.5. |